## _________________________________ **Data Cleanser** _________________________________

### Dataset Generation

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_samples = 500

patient_ids = [f"P{str(i).zfill(4)}" for i in range(1, n_samples + 1)]
age = np.random.randint(18, 80, size=n_samples).astype(float)
gender = np.random.choice(["Male", "Female"], size=n_samples)
region = np.random.choice(["North", "South", "East", "West"], size=n_samples)
bmi = np.random.normal(26.5, 4.5, size=n_samples)
blood_pressure = np.random.normal(120, 15, size=n_samples)
cholesterol = np.random.normal(200, 30, size=n_samples)
glucose = np.random.normal(100, 20, size=n_samples)

logit = (
    -8
    + 0.04 * age
    + 0.08 * bmi
    + 0.02 * blood_pressure
    + 0.01 * cholesterol
    + 0.03 * glucose
)
prob = 1 / (1 + np.exp(-logit))
disease_risk = (prob > np.random.rand(n_samples)).astype(int)

df = pd.DataFrame(
    {
        "patient_id": patient_ids,
        "age": age,
        "gender": gender,
        "region": region,
        "bmi": bmi,
        "blood_pressure": blood_pressure,
        "cholesterol": cholesterol,
        "glucose": glucose,
        "disease_risk": disease_risk,
    }
)

# Injecting Missing Values
df.loc[
    np.random.choice([True, False], size=n_samples, p=[0.08, 0.92]), "age"
] = np.nan
df.loc[
    np.random.choice([True, False], size=n_samples, p=[0.05, 0.95]), "gender"
] = np.nan
df.loc[
    np.random.choice([True, False], size=n_samples, p=[0.06, 0.94]), "region"
] = np.nan
df.loc[
    np.random.choice([True, False], size=n_samples, p=[0.10, 0.90]), "bmi"
] = np.nan
df.loc[
    np.random.choice([True, False], size=n_samples, p=[0.08, 0.92]),
    "cholesterol",
] = np.nan
df.loc[
    np.random.choice([True, False], size=n_samples, p=[0.07, 0.93]), "glucose"
] = np.nan

# Injecting Outliers
df.loc[np.random.choice(df.index, size=5, replace=False), "bmi"] = np.random.uniform(
    60.0, 75.0, size=5
)
df.loc[np.random.choice(df.index, size=8, replace=False), "blood_pressure"] = (
    np.random.uniform(190.0, 240.0, size=8)
)
df.loc[
    np.random.choice(df.index, size=5, replace=False), "cholesterol"
] = np.random.uniform(400.0, 500.0, size=5)
df.loc[np.random.choice(df.index, size=8, replace=False), "glucose"] = (
    np.random.uniform(300.0, 450.0, size=8)
)

df.to_csv("patient_health_records.csv", index=False)
print("Dataset created successfully!")

Dataset created successfully!


### Part A: Handling Missing Values

#### 1. Identify missing values and provide a summary report (percentage per column)

Explanation: In this task, we check the dataset for any missing values (NaN) across all columns and calculate their percentage to understand which columns lack data

In [12]:
import numpy as np
import pandas as pd

df = pd.read_csv("patient_health_records.csv")

# Missing values count ane percentage calculate karo
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Summary Report DataFrame banavo
missing_summary = pd.DataFrame(
    {"Missing Count": missing_count, "Missing Percentage (%)": missing_percentage}
)

# Sort by missing percentage descending
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)", ascending=False
)

print("--- Part A (Task 1): Missing Values Summary Report ---")
print(missing_summary)

--- Part A (Task 1): Missing Values Summary Report ---
                Missing Count  Missing Percentage (%)
bmi                        45                     9.0
age                        41                     8.2
glucose                    35                     7.0
cholesterol                35                     7.0
region                     33                     6.6
gender                     23                     4.6
patient_id                  0                     0.0
blood_pressure              0                     0.0
disease_risk                0                     0.0


#### 2. Apply the following imputation techniques and compare results

#### **( 1 ) Simple Imputer (Numerical):** Replace missing BMI with mean or median

Answer: For the numerical column bmi, we use SimpleImputer with the median strategy because the median is robust against outliers.

In [13]:
from sklearn.impute import SimpleImputer

# Median no use kariye chie kar ke outliers thi median vadhare robust hoy che
num_imputer = SimpleImputer(strategy="median")
df["bmi"] = num_imputer.fit_transform(df[["bmi"]])
print("Simple Imputer (Numerical - BMI) applied successfully!")

Simple Imputer (Numerical - BMI) applied successfully!


#### **( 2 ) Simple Imputer (Categorical):** Replace missing Region with the most frequent value

Answer : For the categorical column region, missing values are replaced with the most frequently occurring value (mode).

In [14]:
region_imputer = SimpleImputer(strategy="most_frequent")
df["region"] = region_imputer.fit_transform(df[["region"]]).ravel()
print("Simple Imputer (Categorical - Region) applied successfully!")

Simple Imputer (Categorical - Region) applied successfully!


#### **( 3 ) Most Frequent Imputation:** Replace missing Gender with the most common category

Answer : The gender column missing entries are filled using the most common category strategy.

In [17]:
gender_imputer = SimpleImputer(strategy="most_frequent")
df["gender"] = pd.Series(
    gender_imputer.fit_transform(df[["gender"]]).ravel(), index=df.index
)
print("Most Frequent Imputation (Gender) applied successfully!")

Most Frequent Imputation (Gender) applied successfully!


#### **( 4 ) Missing Indicator + Random Sample Imputation:** Create binary indicator columns for missingness and use random sampling to impute values

Answer : This technique creates an extra binary column (1 if missing, 0 if present) to retain missingness information, and fills the missing cells using random sampling from existing non-null values

In [18]:
# Applying Missing Indicator + Random Sample Imputation for age and cholesterol
for col in ["age", "cholesterol"]:
  # 1. Create a binary missing indicator column
  df[f"{col}_missing_indicator"] = df[col].isnull().astype(int)

  # 2. Random sample imputation for missing values
  not_null_values = df[col].dropna()
  missing_mask = df[col].isnull()
  df.loc[missing_mask, col] = np.random.choice(
      not_null_values, size=missing_mask.sum()
  )

print(
    "Missing Indicator + Random Sample Imputation applied successfully!"
)

Missing Indicator + Random Sample Imputation applied successfully!


#### **( 5 ) KNN Imputer:** Apply k-Nearest Neighbors imputation for multivariate imputation

Answer : KNN Imputer finds the nearest neighboring patients based on feature similarity and imputes missing values using their average values.

In [19]:
from sklearn.impute import KNNImputer

# Select numerical columns for KNN imputation
knn_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]
knn_imputer = KNNImputer(n_neighbors=5)
df[knn_cols] = knn_imputer.fit_transform(df[knn_cols])
print("KNN Imputer applied successfully!")

KNN Imputer applied successfully!


#### **( 6 ) MICE Algorithm:** Perform chained equations imputation for multiple variables simultaneously

Answer : MICE (Multivariate Imputation by Chained Equations) models each feature with missing values as a function of other features in a round-robin fashion.

In [20]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_cols = ["age", "bmi", "blood_pressure", "cholesterol", "glucose"]

# Apply MICE Algorithm
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df[mice_cols] = mice_imputer.fit_transform(df[mice_cols])

print("MICE Algorithm applied successfully!")

MICE Algorithm applied successfully!


### Part B: Handling Outliers

#### 3. Detect and remove outliers using

##### **-> Z-score method** (For Cholesterol and Glucose)
Explanation: Z-score measures how many standard deviations a data point is from the mean. Values with an absolute Z-score greater than 3 are typically considered extreme outliers and are removed.

In [21]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Make a copy of dataframe for Z-score removal
df_zscore = df.copy()
print(f"Shape before Z-score removal: {df_zscore.shape}")

# Calculate Z-scores for cholesterol and glucose
z_scores = np.abs(
    stats.zscore(df_zscore[["cholesterol", "glucose"]].dropna())
)

# Filter rows where absolute z-score is less than 3
filtered_index = (
    np.abs(
        stats.zscore(df_zscore[["cholesterol", "glucose"]].fillna(0))
    )
    < 3
).all(axis=1)
df_zscore = df_zscore[filtered_index]

print(f"Shape after Z-score removal: {df_zscore.shape}")

Shape before Z-score removal: (500, 11)
Shape after Z-score removal: (487, 11)


##### **-> IQR method** (For BMI)
Explanation: The Interquartile Range (IQR) method uses the 25th (Q1) and 75th (Q3) percentiles. Values below $Q1 - 1.5 \times IQR$ or above $Q3 + 1.5 \times IQR$ are classified as outliers and removed.  

In [22]:
# Make a copy of dataframe for IQR removal
df_iqr = df.copy()
print(f"Shape before IQR removal: {df_iqr.shape}")

# Calculate Q1, Q3, and IQR for BMI
Q1 = df_iqr["bmi"].quantile(0.25)
Q3 = df_iqr["bmi"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out BMI outliers
df_iqr = df_iqr[
    (df_iqr["bmi"] >= lower_bound) & (df_iqr["bmi"] <= upper_bound)
]

print(f"Shape after IQR removal: {df_iqr.shape}")

Shape before IQR removal: (500, 11)
Shape after IQR removal: (484, 11)


##### **-> Percentile method** (Capping below 1st and above 99th percentile)
Explanation: Instead of dropping entire rows, the percentile method caps (clips) extreme values by setting anything below the 1st percentile to the 1st percentile value, and anything above the 99th percentile to the 99th percentile value.

In [23]:
# Make a copy of dataframe for Percentile capping
df_percentile = df.copy()

# Apply percentile capping on continuous features
for col in ["bmi", "cholesterol", "glucose"]:
  p1 = df_percentile[col].quantile(0.01)
  p99 = df_percentile[col].quantile(0.99)
  df_percentile[col] = np.clip(df_percentile[col], p1, p99)

print("Percentile capping applied successfully to BMI, Cholesterol, and Glucose!")

Percentile capping applied successfully to BMI, Cholesterol, and Glucose!


#### 4. Apply Winsorization to cap extreme outliers instead of removing them

Explanation: Winsorization replaces extreme outliers with specific percentiles (like 1st and 99th) instead of deleting rows, which helps preserve the sample size

In [24]:
from scipy.stats import mstats

# Applying Winsorization to numerical columns to cap extreme values
cols_to_winsorize = ["bmi", "blood_pressure", "cholesterol", "glucose"]

for col in cols_to_winsorize:
  # Limits set to 1% on both lower and upper tails
  df[col] = mstats.winsorize(df[col], limits=[0.01, 0.01])

print("Winsorization applied successfully to all continuous columns!")

Winsorization applied successfully to all continuous columns!


#### 5. Compare dataset shape and summary before vs after outlier treatment


In [25]:
# Comparison of Dataset Shapes and Descriptive Statistics
print("--- Before Outlier Treatment Summary ---")
print(df[["bmi", "blood_pressure", "cholesterol", "glucose"]].describe())

# Applying winsorization/cleaning to main dataframe
for col in ["bmi", "blood_pressure", "cholesterol", "glucose"]:
  df[col] = mstats.winsorize(df[col], limits=[0.01, 0.01])

print("\n--- After Outlier Treatment Summary ---")
print(df[["bmi", "blood_pressure", "cholesterol", "glucose"]].describe())

--- Before Outlier Treatment Summary ---
              bmi  blood_pressure  cholesterol     glucose
count  500.000000      500.000000   500.000000  500.000000
mean    26.590295      121.158805   200.707474  103.470612
std      4.375845       18.368644    30.036568   37.086507
min     16.445081       85.854397   133.725451   52.174511
25%     23.852848      109.394539   180.413097   86.278319
50%     26.503146      120.485442   200.190182  100.078961
75%     29.129018      130.069489   221.529390  112.895807
max     39.842547      211.925401   281.926017  356.197383

--- After Outlier Treatment Summary ---
              bmi  blood_pressure  cholesterol     glucose
count  500.000000      500.000000   500.000000  500.000000
mean    26.590295      121.158805   200.707474  103.470612
std      4.375845       18.368644    30.036568   37.086507
min     16.445081       85.854397   133.725451   52.174511
25%     23.852848      109.394539   180.413097   86.278319
50%     26.503146      120.485442

c:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4798: UserWarning: Warning: 'partition' will ignore the 'mask' of the Maske

### Part C: Final Clean Dataset & Report

#### 6. Present the final cleaned dataset with

##### -> All missing values treated appropriately.

In [27]:
# Create a copy of the dataframe for treating missing values
df_missing_treated = df.copy()

# Filling missing values for numerical and categorical columns
df_missing_treated["bmi"].fillna(
    df_missing_treated["bmi"].median(), inplace=True
)
df_missing_treated["region"].fillna(
    df_missing_treated["region"].mode()[0], inplace=True
)
df_missing_treated["gender"].fillna(
    df_missing_treated["gender"].mode()[0], inplace=True
)
df_missing_treated["age"].fillna(
    df_missing_treated["age"].median(), inplace=True
)
df_missing_treated["cholesterol"].fillna(
    df_missing_treated["cholesterol"].median(), inplace=True
)
df_missing_treated["glucose"].fillna(
    df_missing_treated["glucose"].median(), inplace=True
)

print("--- Sub-point 1: Missing Values Treated Successfully ---")
print(df_missing_treated.isnull().sum())


--- Sub-point 1: Missing Values Treated Successfully ---
patient_id                       0
age                              0
gender                           0
region                           0
bmi                              0
blood_pressure                   0
cholesterol                      0
glucose                          0
disease_risk                     0
age_missing_indicator            0
cholesterol_missing_indicator    0
dtype: int64


c:\Users\dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\_core\fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)
C:\Users\dell\AppData\Local\Temp\ipykernel_3344\2142011419.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_wr

##### -> All outliers handled using suitable methods.

In [28]:
from scipy.stats import mstats

# Use the dataframe where missing values are already treated
df_final_clean = df_missing_treated.copy()

# Handling outliers using Winsorization (Capping at 1st and 99th percentiles)
for col in ["bmi", "blood_pressure", "cholesterol", "glucose"]:
  df_final_clean[col] = mstats.winsorize(df_final_clean[col], limits=[0.01, 0.01])

# Save the final clean dataset to CSV
df_final_clean.to_csv("cleaned_patient_health_records.csv", index=False)

print("--- Sub-point 2: Outliers Handled Successfully ---")
print(df_final_clean.head())
print(
    "\nFinal clean dataset successfully saved as 'cleaned_patient_health_records.csv'!"
)

--- Sub-point 2: Outliers Handled Successfully ---
  patient_id   age  gender region        bmi  blood_pressure  cholesterol  \
0      P0001  56.0  Female  South  21.588743      117.996302   211.384462   
1      P0002  69.0    Male  South  26.503146      136.287995   254.345144   
2      P0003  46.0  Female   West  33.293971       88.486217   186.137119   
3      P0004  32.0    Male  South  28.309155      164.066293   175.129492   
4      P0005  60.0    Male  South  24.936130      106.932994   162.009785   

      glucose  disease_risk  age_missing_indicator  \
0  103.845860             1                      0   
1   97.035542             1                      0   
2   82.430343             1                      0   
3   71.749285             1                      0   
4   99.425820             1                      0   

   cholesterol_missing_indicator  
0                              0  
1                              0  
2                              1  
3                    

#### 7. Provide a brief report explaining


# Data Cleanser: Project Summary Report

### 1. Most Effective Imputation Strategy:
* **KNN Imputer and MICE Algorithm** were the most effective for numerical features (`bmi`, `cholesterol`, `glucose`) because they preserve multivariate correlations between patient vitals instead of replacing values blindly with a flat mean/median[cite: 3].
* For categorical variables (`region`, `gender`), **Most Frequent (Mode) Imputation** maintained the original distribution balance effectively[cite: 3].

### 2. Best Outlier Handling Method:
* **Winsorization (Percentile Capping at 1st and 99th percentiles)** preserved data quality the best[cite: 4]. Unlike traditional deletion methods (like Z-score or IQR) which drop entire patient rows and discard valuable clinical details, Winsorization successfully mitigated extreme measurement spikes while keeping sample sizes fully intact for downstream machine learning tasks.

### 3. Impact of Data Cleaning on Dataset Usability:
* Resolving missing entries and smoothing extreme synthetic anomalies eliminated model bias, prevented skewed distributions, and ensured feature stability.
* The finalized clean dataset (`cleaned_patient_health_records.csv`) is now fully standardized, robust, and optimal for building predictive models (such as predicting `disease_risk`)[cite: 1, 3, 4].